In [17]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 5.9 MB/s eta 0:00:00


In [42]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold,RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
import pickle
from catboost import CatBoostClassifier

In [52]:
df_all=pd.read_csv('umap_all_data.csv')
df_umap=pd.read_csv('umap_target_data.csv')

In [54]:
#отделяем признаки от целевой переменной и делаем ее логарифмирование
targets = ['IC50, mM', 'CC50, mM', 'SI']
all_features = [col for col in df_all.columns if col not in targets]
X_all = df_all[all_features].copy()
all_features2 = [col for col in df_umap.columns if col not in targets]
X_umap = df_umap[all_features2].copy()
y_a = df_all['SI']
y_u = df_umap['SI']

In [55]:
#делим выборку на обучающую и валидационную
X_a_train, X_a_test, y_a_train, y_a_test = train_test_split(X_all, y_a, test_size=0.2, random_state=5)
X_u_train, X_u_test, y_u_train, y_u_test = train_test_split(X_umap, y_u, test_size=0.2, random_state=5)


In [56]:
#делаем разметку классов больше/меньше 8 в train и test выборках

y_a_train_cls = (y_a_train > 8).astype(int)
y_a_test_cls = (y_a_test > 8).astype(int)

y_u_train_cls = (y_u_train > 8).astype(int)
y_u_test_cls = (y_u_test > 8).astype(int)

In [57]:
print('Распределение целевой переменной в обучающей выборке')
print(y_a_train_cls.value_counts())
print('Распределение целевой переменной в валидационной выборке')
print(y_a_test_cls.value_counts())

Распределение целевой переменной в обучающей выборке
SI
0    511
1    287
Name: count, dtype: int64
Распределение целевой переменной в валидационной выборке
SI
0    131
1     69
Name: count, dtype: int64


In [62]:
#классы не сбалансиованны, сделаем SMOTE
smote = SMOTE(random_state=5, k_neighbors=5)
X_a_train_res, y_a_train_res = smote.fit_resample(X_a_train, y_a_train_cls)
y_a_train_cls=y_a_train_res.copy()

In [63]:
scaler_robust = RobustScaler()
X_a_train_r = pd.DataFrame(scaler_robust.fit_transform(X_a_train_res), columns=X_a_train_res.columns, index=X_a_train_res.index)
X_a_test_r = pd.DataFrame(scaler_robust.transform(X_a_test), columns=X_a_test.columns, index=X_a_test.index)

In [64]:
#выбираем нелинейные модели для обучения
#определяем cv для кроссвалидации
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=5)
models = {'Random Forest': {'model': RandomForestClassifier(random_state=5, n_jobs=-1),
        'params': {'n_estimators': [200, 300, 350],
                   'max_depth': [10, 20, 30, None],
                   'min_samples_split': [5, 10, 15, 20],
                   'min_samples_leaf': [2, 4, 5]}},
          'Gradient Boosting': {'model': GradientBoostingClassifier(random_state=5),
        'params': {'n_estimators': [200, 300],
                   'learning_rate': [0.01, 0.05, 0.1],
                   'max_depth': [2, 3, 4],
                   'subsample': [0.6, 0.8, 1.0]}},
          'XGBoost': {'model': xgb.XGBClassifier(random_state=5, eval_metric='logloss'),
        'params': {'n_estimators': [200, 250],
            'max_depth': [2, 3, 5, 7],
            'learning_rate': [0.01, 0.05, 0.2]}},
          'LightGBM': {'model': lgb.LGBMClassifier(random_state=5, verbose=-1),
        'params': {'n_estimators': [100, 150],
            'num_leaves': [10, 15, 30],
            'learning_rate': [0.05, 0.1, 0.2]}},
          'SVM': {'model': SVC(random_state=5, probability=True),
        'params': {'C': [1, 10, 30, 50],
                   'gamma': ['scale', 'auto', 0.1, 1],
                   'kernel': ['rbf']}},
          'KNN': {'model': KNeighborsClassifier(),
        'params': {'n_neighbors': [3, 5, 7, 11, 15],
                    'weights': ['uniform', 'distance'],
                   'metric': ['euclidean', 'manhattan']}}}


In [66]:
#обучаем модели и сохраняем лучшие результаты
results = []
best_models = {}
print('Модели на всех признаках')
for name, config in models.items():
  grid = GridSearchCV(config['model'], config['params'],
        cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0)
  grid.fit(X_a_train_r, y_a_train_cls)

  best_model = grid.best_estimator_
  best_models[name] = best_model

  y_pred = best_model.predict(X_a_test_r)
  y_pred_proba = best_model.predict_proba(X_a_test_r)[:, 1]
  results.append({'Model': name,
        'Best Params': str(grid.best_params_),
        'Accuracy': accuracy_score(y_a_test_cls, y_pred),
        'F1-Score': f1_score(y_a_test_cls, y_pred),
        'ROC-AUC': roc_auc_score(y_a_test_cls, y_pred_proba),
        'CV Score': grid.best_score_})
  print(f'\n{name}')
  print(f'Лучшие параметры: {grid.best_params_}')
  print(f'Accuracy: {accuracy_score(y_a_test_cls, y_pred):.3f}')
  print(f'ROC-AUC:  {roc_auc_score(y_a_test_cls, y_pred_proba):.3f}')
  print(f'F1-Score: {f1_score(y_a_test_cls, y_pred):.3f}')

Модели на всех признаках

Random Forest
Лучшие параметры: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 350}
Accuracy: 0.680
ROC-AUC:  0.725
F1-Score: 0.500

Gradient Boosting
Лучшие параметры: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 300, 'subsample': 0.8}
Accuracy: 0.665
ROC-AUC:  0.704
F1-Score: 0.472

XGBoost
Лучшие параметры: {'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 250}
Accuracy: 0.660
ROC-AUC:  0.722
F1-Score: 0.500

LightGBM
Лучшие параметры: {'learning_rate': 0.05, 'n_estimators': 150, 'num_leaves': 30}
Accuracy: 0.700
ROC-AUC:  0.741
F1-Score: 0.552

SVM
Лучшие параметры: {'C': 30, 'gamma': 'auto', 'kernel': 'rbf'}
Accuracy: 0.685
ROC-AUC:  0.704
F1-Score: 0.512

KNN
Лучшие параметры: {'metric': 'manhattan', 'n_neighbors': 5, 'weights': 'distance'}
Accuracy: 0.640
ROC-AUC:  0.686
F1-Score: 0.507


In [67]:
results = []
best_models = {}
print('Модели на UMAP признаках')
for name, config in models.items():
  grid = GridSearchCV(config['model'], config['params'],
        cv=cv, scoring='roc_auc', n_jobs=-1, verbose=0)
  grid.fit(X_u_train, y_u_train_cls)

  best_model = grid.best_estimator_
  best_models[name] = best_model

  y_pred = best_model.predict(X_u_test)
  y_pred_proba = best_model.predict_proba(X_u_test)[:, 1]
  results.append({'Model': name,
        'Best Params': str(grid.best_params_),
        'Accuracy': accuracy_score(y_u_test_cls, y_pred),
        'F1-Score': f1_score(y_u_test_cls, y_pred),
        'ROC-AUC': roc_auc_score(y_u_test_cls, y_pred_proba),
        'CV Score': grid.best_score_})
  print(f'\n{name}')
  print(f'Лучшие параметры: {grid.best_params_}')
  print(f'Accuracy: {accuracy_score(y_u_test_cls, y_pred):.3f}')
  print(f'ROC-AUC:  {roc_auc_score(y_u_test_cls, y_pred_proba):.3f}')
  print(f'F1-Score: {f1_score(y_u_test_cls, y_pred):.3f}')

Модели на UMAP признаках

Random Forest
Лучшие параметры: {'max_depth': 20, 'min_samples_leaf': 5, 'min_samples_split': 15, 'n_estimators': 200}
Accuracy: 0.715
ROC-AUC:  0.712
F1-Score: 0.477

Gradient Boosting
Лучшие параметры: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 1.0}
Accuracy: 0.695
ROC-AUC:  0.679
F1-Score: 0.384

XGBoost
Лучшие параметры: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200}
Accuracy: 0.685
ROC-AUC:  0.690
F1-Score: 0.337

LightGBM
Лучшие параметры: {'learning_rate': 0.05, 'n_estimators': 100, 'num_leaves': 10}
Accuracy: 0.705
ROC-AUC:  0.710
F1-Score: 0.468

SVM
Лучшие параметры: {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}
Accuracy: 0.700
ROC-AUC:  0.746
F1-Score: 0.444

KNN
Лучшие параметры: {'metric': 'euclidean', 'n_neighbors': 11, 'weights': 'uniform'}
Accuracy: 0.705
ROC-AUC:  0.725
F1-Score: 0.478


В целом модели только на UMAP признаках дают результаты хуже, поэтому продолжим оптимизацию с данными со всеми выбранными признаками

In [69]:
#оптимизируем LightGBM модель
lgbm_params = {
    'num_leaves': [8, 10, 12, 15],
     'n_estimators': [100, 120, 140],
    'learning_rate': [0.08, 0.1, 0.15, 0.2],
    'max_depth': [5, 7, 10],
    'subsample': [0.8, 1.0],
    'min_child_samples': [10, 15, 20, 30]}
random_lgbm = RandomizedSearchCV(lgb.LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1), lgbm_params, n_iter=50,
                                 cv=cv, scoring='balanced_accuracy', random_state=42, n_jobs=-1, verbose=1)
random_lgbm.fit(X_a_train_r, y_a_train_cls)
print(f'Best params: {random_lgbm.best_params_}')
best_r_lgbm = random_lgbm.best_estimator_
y_pred_r_lgbm = best_r_lgbm.predict(X_a_test_r)
y_pred_proba_r_lgbm = best_r_lgbm.predict_proba(X_a_test_r)[:, 1]

print(f'Accuracy:  {accuracy_score(y_a_test_cls, y_pred_r_lgbm):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_a_test_cls, y_pred_proba_r_lgbm):.4f}')
print(f'F1-Score:  {f1_score(y_a_test_cls, y_pred_r_lgbm):.4f}')

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'subsample': 0.8, 'num_leaves': 10, 'n_estimators': 100, 'min_child_samples': 15, 'max_depth': 10, 'learning_rate': 0.08}
Accuracy:  0.7100
ROC-AUC:   0.7409
F1-Score:  0.5797


In [78]:
#обучаем алгоритм Catboost
catboost_base = CatBoostClassifier(
    iterations=500,
    learning_rate=0.03,
    depth=6,
    auto_class_weights='Balanced',
    cat_features=[],
    random_seed=36,
    eval_metric='F1')

catboost_base.fit( X_a_train_r, y_a_train_cls,
    eval_set=(X_a_test_r , y_a_test_cls),
    verbose=100,
    plot=False)

y_pred_base = catboost_base.predict(X_a_test_r)
y_pred_proba_base = catboost_base.predict_proba(X_a_test_r)[:, 1]

print(f"Accuracy:  {accuracy_score(y_a_test_cls, y_pred_base):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_a_test_cls, y_pred_proba_base):.4f}")
print(f"F1-Score:  {f1_score(y_a_test_cls, y_pred_base):.4f}")

0:	learn: 0.6729758	test: 0.4571429	best: 0.4571429 (0)	total: 19.7ms	remaining: 9.83s
100:	learn: 0.8293173	test: 0.5588235	best: 0.5777778 (79)	total: 1.55s	remaining: 6.12s
200:	learn: 0.8888889	test: 0.6029412	best: 0.6330935 (166)	total: 2.77s	remaining: 4.12s
300:	learn: 0.9194956	test: 0.5076923	best: 0.6330935 (166)	total: 4.12s	remaining: 2.72s
400:	learn: 0.9513619	test: 0.4769231	best: 0.6330935 (166)	total: 5.42s	remaining: 1.34s
499:	learn: 0.9725490	test: 0.4603175	best: 0.6330935 (166)	total: 6.86s	remaining: 0us

bestTest = 0.6330935252
bestIteration = 166

Shrink model to first 167 iterations.
Accuracy:  0.7450
ROC-AUC:   0.7429
F1-Score:  0.6331


In [71]:
cat_params = {
   'iterations': [300, 500, 1000],
    'learning_rate': [0.005, 0.01, 0.02, 0.03],
    'depth': [4, 5, 6, 7],
    'l2_leaf_reg': [2, 3, 4, 12],
    'border_count': [128, 255],
    'auto_class_weights': ['Balanced'],
   'random_strength': [0.5, 1, 2]}
random_cat = RandomizedSearchCV(
    CatBoostClassifier(random_seed=5, verbose=0, eval_metric='AUC'),
    param_distributions=cat_params,
    n_iter=50, cv=cv,scoring='roc_auc',n_jobs=-1,random_state=5, verbose=1)
random_cat.fit(X_a_train_r, y_a_train_cls)
best_cat = random_cat.best_estimator_
y_pred_cat = best_cat.predict(X_a_test_r)
y_pred_proba_cat = best_cat.predict_proba(X_a_test_r)[:, 1]
print(f'Best params: {random_cat.best_params_}')
print(f"Accuracy: {accuracy_score(y_a_test_cls, y_pred_cat):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_a_test_cls, y_pred_proba_cat):.4f}")
print(f"F1-Score: {f1_score(y_a_test_cls, y_pred_cat):.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'random_strength': 1, 'learning_rate': 0.03, 'l2_leaf_reg': 2, 'iterations': 500, 'depth': 6, 'border_count': 128, 'auto_class_weights': 'Balanced'}
Accuracy: 0.6900
ROC-AUC:  0.7275
F1-Score: 0.5303


In [79]:
#сохраняем полученную модель
with open('catboost_base_SI_8.pkl', 'wb') as f:
    pickle.dump(catboost_base, f)
with open('scaler_SI_8.pkl', 'wb') as f:
    pickle.dump(scaler_robust, f)
print('Модель и компоненты сохранены в сессионное хранилище')

Модель и компоненты сохранены в сессионное хранилище


In [80]:
#проверяем загрузку
with open('catboost_base_SI_8.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
with open('scaler_SI_8.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)
print('Модель и компоненты загружены')

Модель и компоненты загружены
